# sincronizar-evento-titulo-tags.ipynb — Copia as tags de evento/título pra planilha

As tags de **evento** e **título** já existem prontas em `dados_lexico/eventos-biblicos.json`/`titulos-biblicos.json` (calculadas sem IA, léxico + dicionário de sinônimo) -- mas ficavam **escondidas** dentro desses arquivos, sem aparecer na planilha.

Esse notebook copia isso pra 2 abas novas na **Biblioteca de Match** -- `evento_tags` e `titulo_tags` -- com o **intervalo** de cada um bem visível (capítulo pro evento, capítulo+versículo pro título).

**Roda a Bíblia inteira de uma vez** (é grátis, sem IA nem Pixabay) -- e não duplica se você rodar de novo depois (só adiciona o que ainda não tinha sido copiado, ex: se você atualizar o léxico com mais sinônimo).

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1️⃣  SETUP                                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread "mistralai>=1.2.0"

from google.colab import drive, auth, userdata
from google.auth import default
import gspread

drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

import shutil
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
_pasta_modulos = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos"
for _arquivo in Path(_pasta_modulos).glob("*.py"):
    shutil.copy(_arquivo, ".")

from groq import Groq
from mistralai.client import Mistral

GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None
MODELO_GROQ = "qwen/qwen3.6-27b"
MODELO_MISTRAL = "mistral-small-latest"

print("✅ Setup pronto")
print(f"   Groq:    {'disponível' if groq_client else 'não configurado'}")
print(f"   Mistral: {'disponível' if mistral_client else 'não configurado'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup pronto
   Groq:    disponível
   Mistral: disponível


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# Mesmo ID da Biblioteca de Match que os outros notebooks já usam
ID_PLANILHA_BIBLIOTECA_MATCH = "1i67VxksAkWYx1cZ_QeoesGXsW28hcA0p5IIfhjx8VHE"

NOME_ARQUIVO_EVENTOS = "eventos-biblicos.json"
NOME_ARQUIVO_TITULOS = "titulos-biblicos.json"
PASTA_DADOS_LEXICO = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/dados_lexico"

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Biblioteca de Match: {ID_PLANILHA_BIBLIOTECA_MATCH}")
print("=" * 60)

⚙️  CONFIGURAÇÃO
   Biblioteca de Match: 1i67VxksAkWYx1cZ_QeoesGXsW28hcA0p5IIfhjx8VHE


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  CARREGAR LÉXICO + ABRIR PLANILHA                            ║
# ╚══════════════════════════════════════════════════════════════════╝
import json
from drive_utils import DriveClient
from match_pipeline import (
    abrir_ou_criar_biblioteca_match, garantir_aba_evento_tags, garantir_aba_titulo_tags,
    sincronizar_evento_titulo_tags,
)
from trilha_pipeline import sugerir_tags_clima_eventos_em_lote, gravar_tags_clima

_drive = DriveClient.get()

_dest_eventos = Path(NOME_ARQUIVO_EVENTOS)
_drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_EVENTOS, _dest_eventos)
with open(_dest_eventos, encoding="utf-8") as _f:
    eventos_biblicos = json.load(_f)

_dest_titulos = Path(NOME_ARQUIVO_TITULOS)
_drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_TITULOS, _dest_titulos)
with open(_dest_titulos, encoding="utf-8") as _f:
    titulos_biblicos = json.load(_f)

print(f"✅ Léxico: {len(eventos_biblicos)} eventos, {len(titulos_biblicos)} títulos")

_spreadsheet_biblioteca, _aba_biblioteca_match, _ = abrir_ou_criar_biblioteca_match(
    gc, ID_PLANILHA_BIBLIOTECA_MATCH, "biblioteca_match",
)
aba_evento_tags = garantir_aba_evento_tags(_spreadsheet_biblioteca)
aba_titulo_tags = garantir_aba_titulo_tags(_spreadsheet_biblioteca)
print("✅ Abas evento_tags/titulo_tags abertas/criadas")

✅ Léxico: 511 eventos, 2858 títulos
✅ Abas evento_tags/titulo_tags abertas/criadas


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3️⃣  SINCRONIZAR — Bíblia inteira, grátis, não duplica           ║
# ╚══════════════════════════════════════════════════════════════════╝
n_eventos, n_titulos = sincronizar_evento_titulo_tags(
    aba_evento_tags, aba_titulo_tags, eventos_biblicos, titulos_biblicos,
)
print(f"✅ {n_eventos} evento(s) e {n_titulos} título(s) copiados pra planilha")
if n_eventos == 0 and n_titulos == 0:
    print("   (nada novo -- já estava tudo sincronizado)")

✅ 0 evento(s) e 0 título(s) copiados pra planilha
   (nada novo -- já estava tudo sincronizado)


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4️⃣  SUGERIR CLIMA (IA) — só quem ainda não tem tags_clima        ║
# ╚══════════════════════════════════════════════════════════════════╝
# A IA sugere, você revisa/corrige direto na planilha depois -- não é
# a palavra final. Roda em lote (não 1 chamada por evento) e nunca
# sobrescreve o que você já corrigiu na mão (ver gravar_tags_clima()).

if not (groq_client or mistral_client):
    print("⚠️  Nenhuma API de IA disponível -- pulando sugestão de clima.")
else:
    _registros_evento_tags = aba_evento_tags.get_all_records()
    _ja_tem_clima = {str(r["evento_id"]) for r in _registros_evento_tags if str(r.get("tags_clima", "")).strip()}
    _eventos_sem_clima = [e for e in eventos_biblicos if e.get("id") not in _ja_tem_clima]

    print(f"📋 {len(_eventos_sem_clima)} evento(s) sem clima ainda (de {len(eventos_biblicos)} no total)")
    if _eventos_sem_clima:
        _estado_provedor_clima = {"atual": "mistral"}
        climas_sugeridos = sugerir_tags_clima_eventos_em_lote(
            _eventos_sem_clima, groq_client, mistral_client, _estado_provedor_clima,
            MODELO_GROQ, MODELO_MISTRAL, tamanho_lote=15,
        )
        n_atualizados = gravar_tags_clima(aba_evento_tags, climas_sugeridos, coluna_id="evento_id")
        print(f"✅ {n_atualizados} evento(s) com clima sugerido -- revise/corrija direto na planilha evento_tags")
    else:
        print("Nada pra sugerir -- todos os eventos já têm clima (sugerido ou corrigido).")


📋 0 evento(s) sem clima ainda (de 511 no total)
Nada pra sugerir -- todos os eventos já têm clima (sugerido ou corrigido).


In [6]:
import trilha_pipeline, inspect
tem_correcao = 'AUTOCURA' in inspect.getsource(trilha_pipeline.gravar_tags_clima)
print("Correção carregada?", tem_correcao)

Correção carregada? True


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5️⃣  SUGERIR CLIMA (IA) — TÍTULO — só quem ainda não tem          ║
# ╚══════════════════════════════════════════════════════════════════╝
# Mesma lógica da célula anterior, agora pro TÍTULO (granularidade de
# versículo -- mais fino que evento). Título é o que o versículo usa
# PRIMEIRO (se tiver clima) -- só cai pro clima do evento se o título
# que cobre esse versículo não tiver clima próprio.

if not (groq_client or mistral_client):
    print("⚠️  Nenhuma API de IA disponível -- pulando sugestão de clima (título).")
else:
    _registros_titulo_tags = aba_titulo_tags.get_all_records()
    _titulos_ja_tem_clima = {str(r["titulo_id"]) for r in _registros_titulo_tags if str(r.get("tags_clima", "")).strip()}
    # titulos_biblicos é um dict {chave_id: entrada} -- vira lista de {id, titulo} pra reaproveitar a mesma função
    _titulos_para_sugerir = [
        {"id": chave_id, "titulo": entrada.get("titulo", "")}
        for chave_id, entrada in titulos_biblicos.items()
        if chave_id not in _titulos_ja_tem_clima
    ]

    print(f"📋 {len(_titulos_para_sugerir)} título(s) sem clima ainda (de {len(titulos_biblicos)} no total)")
    if _titulos_para_sugerir:
        _estado_provedor_clima_titulo = {"atual": "mistral"}
        climas_sugeridos_titulo = sugerir_tags_clima_eventos_em_lote(
            _titulos_para_sugerir, groq_client, mistral_client, _estado_provedor_clima_titulo,
            MODELO_GROQ, MODELO_MISTRAL, tamanho_lote=15,
        )
        n_atualizados_titulo = gravar_tags_clima(aba_titulo_tags, climas_sugeridos_titulo, coluna_id="titulo_id")
        print(f"✅ {n_atualizados_titulo} título(s) com clima sugerido -- revise/corrija direto na planilha titulo_tags")
    else:
        print("Nada pra sugerir -- todos os títulos já têm clima (sugerido ou corrigido).")


📋 2858 título(s) sem clima ainda (de 2858 no total)
   [1/191] 15 evento(s) -- ✅
   [2/191] 15 evento(s) -- ✅
   [3/191] 15 evento(s) -- ✅
   [4/191] 15 evento(s) -- ✅
   [5/191] 15 evento(s) -- ✅
   [6/191] 15 evento(s) -- ✅
   [7/191] 15 evento(s) -- ✅
   [8/191] 15 evento(s) -- ✅
   [9/191] 15 evento(s) -- ✅
   [10/191] 15 evento(s) -- ✅
   [11/191] 15 evento(s) -- ✅
   [12/191] 15 evento(s) -- ✅
   [13/191] 15 evento(s) -- ✅
   [14/191] 15 evento(s) -- ✅
   [15/191] 15 evento(s) -- ✅
   [16/191] 15 evento(s) -- ✅
   [17/191] 15 evento(s) -- ✅
   [18/191] 15 evento(s) -- ✅
   [19/191] 15 evento(s) -- ✅
   [20/191] 15 evento(s) -- ✅
   [21/191] 15 evento(s) -- ✅
   [22/191] 15 evento(s) -- ✅
   [23/191] 15 evento(s) -- ✅
   [24/191] 15 evento(s) -- ✅
   [25/191] 15 evento(s) -- ✅
   [26/191] 15 evento(s) -- ✅
   [27/191] 15 evento(s) -- ✅
   [28/191] 15 evento(s) -- ✅
   [29/191] 15 evento(s) -- ✅
   [30/191] 15 evento(s) -- ✅
   [31/191] 15 evento(s) -- ✅
   [32/191] 15 evento(s) --